# Data Sources and Inputs

The workflow depends on several source families: environmental products, fishing activity, telemetry records, and reference layers. This section explains what each source contributes, where the active configuration lives, and which reusable derived products are archived separately from the Git repository.


## Input Families

- Environmental data: SST, chlorophyll-a, SSH, wind, and bathymetry.
- Fisheries data: Global Fishing Watch AIS-derived apparent fishing activity.
- Biological data: SAERI telemetry records for BBAL and SAFS.
- Reference data: fisheries grid, FICZ/FOCZ, Natural Earth land/coastline.

In the Falkland case study, environmental and fisheries products cover 2014-2023, while telemetry records used here were collected during 2022-2023.


In [1]:
from pathlib import Path
import yaml

config = yaml.safe_load(Path("../config.yaml").read_text())
for name, spec in config.get("datasets", {}).items():
    role = spec.get("role", "unspecified")
    provider = spec.get("provider", "manual/local")
    print(f"{name:12s} role={role:16s} provider={provider}")


sst          role=environmental    provider=podaac
chl          role=environmental    provider=copernicus
ssh          role=environmental    provider=copernicus
wind         role=environmental    provider=cds
gfw          role=fishing_effort   provider=gfw
bathymetry   role=static           provider=gebco


## Repository Structure

The Git repository contains the reusable workflow code, configuration, notebooks, documentation, and lightweight metadata.

The main folders have distinct roles:

- `src/riskscape/`: reusable Python package modules.
- `scripts/`: command-line entry points grouped by task (`data/`, `build/`, `model/`, `plots/`, `qa/`, and `tools/`).
- `notebooks/`: reader-facing workflow demonstrations and inspection notebooks.
- `docs/`: supporting documentation, inventories, and release notes.
- `reference/`: restored public reference layers and their local README.
- `data/`: local derived data products, restored from Zenodo or regenerated by the workflow.
- `plots/`: generated figures and videos, restored from Zenodo or regenerated by plotting scripts.

The main layout is:

```text
falkland-bycatch-riskscape/
├── config.yaml
├── pyproject.toml
├── README.md
├── CITATION.cff
├── LICENSE
├── src/
│   └── riskscape/
├── scripts/
│   ├── data/
│   ├── build/
│   ├── model/
│   ├── plots/
│   ├── qa/
│   └── tools/
├── notebooks/
├── docs/
├── reference/
├── data/
│   ├── grids/
│   ├── processed/
│   ├── features/
│   ├── modeling/
│   ├── plot_exports/
│   └── raw/
└── plots/
```

Key root files include `config.yaml`, `.env.example`, `pyproject.toml`, `CITATION.cff`, `LICENSE`, and the repository `README.md`. The source-controlled parts are the code, notebooks, docs, configuration, and lightweight metadata. The `data/`, `plots/`, and downloaded `reference/` contents are local products restored from Zenodo, restored from public providers, or regenerated by the workflow.


## Raw Data

Raw provider downloads are local working inputs and are not tracked in Git or redistributed in the Zenodo bundle. They may require credentials, provider access approval, or redistribution checks.

To restore or rebuild raw inputs, start with these files:

- `config.yaml`: active dataset names, providers, products, variables, date range, and `data/raw/` location.
- `docs/datasets.md`: source notes and provider links for each dataset family.
- `docs/authentication.md`: credential setup for Copernicus Marine, CDS/ERA5, Global Fishing Watch, and CEDA/GEBCO.
- `.env.example`: local environment-variable names such as `GFW_TOKEN` and CEDA token fields.

Configured provider downloads are run with:

```bash
python3 scripts/data/download_data.py
```

Selected datasets can be downloaded by name, for example:

```bash
python3 scripts/data/download_data.py --dataset sst chl ssh wind gfw bathymetry
```

Downloaded files are written under `data/raw/<dataset>/`. Source species-observation data are not automatically redistributed; place the approved local file in the raw species-presence location expected by the species feature builder before rebuilding species features.


## Reference Layers

Reference layers are public spatial layers used for context, grid definition, and plotting. They are not committed as source files, but they can be restored into `reference/` with:

```bash
python3 scripts/download_reference_data.py
```

The restored reference set includes Natural Earth land and coastline layers plus Falkland fisheries-grid and conservation-zone layers where direct download access is available. The local `reference/README.md` records source notes and any manual steps that remain necessary.


## Zenodo Data Bundle

Reusable derived products are archived separately from the Git repository in the Zenodo data bundle:

<https://doi.org/10.5281/zenodo.20334806>

The Zenodo record is organized as one README plus six compressed archives.

**Bundle README**

`README.md` — 2.9 kB  
Bundle description, citation, license, and restoration notes.

**Data archives**

`falkland-bycatch-riskscape-grids-v0.1.0.tar.gz` — 17.6 MB  
H3 study grid products.

`falkland-bycatch-riskscape-processed-v0.1.0.tar.gz` — 116.4 MB  
Processed lookup/index tables used by feature construction.

`falkland-bycatch-riskscape-features-v0.1.0.tar.gz` — 4.8 GB  
Engineered feature tables.

`falkland-bycatch-riskscape-modeling-v0.1.0.tar.gz` — 11.7 GB  
Modeling products, selected predictions, environmental regimes, plausibility outputs, metrics, and operational tables.

**Plot archives**

`falkland-bycatch-riskscape-plot_exports-v0.1.0.tar.gz` — 66.0 MB  
Lightweight tabular exports used by selected diagnostic plots.

`falkland-bycatch-riskscape-plots-v0.1.0.tar.gz` — 580.2 MB  
Generated figures and videos for inspection and examples.

To restore the bundle into a local checkout, extract the archives and copy their top-level folders into the repository root:

- `grids/`, `processed/`, `features/`, `modeling/`, and `plot_exports/` belong under `data/`.
- `plots/` belongs at the repository root as `plots/`.
- `README.md` can remain next to the downloaded archives as the data-bundle readme.

After restoration, the local paths should look like `data/grids/`, `data/processed/`, `data/features/`, `data/modeling/`, `data/plot_exports/`, and `plots/`.

These archives restore the reusable products needed for inspection, examples, and selected reruns. They are not a complete mirror of every local working folder, and they do not include raw provider downloads or credentials.


## Products Not Redistributed

The Zenodo bundle is not a mirror of every local file. It is intended to provide the reusable derived inputs needed to inspect and rerun the public workflow without redistributing source material that should be obtained from its original provider.

Not included in the data bundle:

- raw provider downloads under `data/raw/`
- local credentials or authentication files
- reference layers that can be restored with `scripts/download_reference_data.py`
- intermediate diagnostics and exploratory products that are regenerated by the build or plotting scripts
- source datasets with sensitive, restricted, or provider-dependent redistribution terms

The retained `modeling/seascape_species_use/` product is limited to the 2022 surface used by the public seascape-conditioned matrices. It can be regenerated with:

```bash
python3 scripts/build/build_seascape_species_use_surfaces.py --years 2022
```


## Credentials

Provider credentials must remain outside committed files. See `docs/authentication.md` for the current convention: Earthdata in `~/.netrc`, Copernicus Marine through `copernicusmarine login`, CDS in `~/.cdsapirc`, and Global Fishing Watch in local `.env` as `GFW_TOKEN`.
